# Libraries and set up

In [1]:
import os, glob, json, random
import torch
import numpy as np
import copy
import time
from sklearn import preprocessing

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from scipy.special import softmax

import matplotlib.pyplot as plt
from matplotlib.patches import Patch

In [2]:
def set_seeds(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    # If using GPU, this ensures deterministic operations (can be slower)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

SEED=42
set_seeds(SEED)

In [3]:
# This automatically picks GPU if available, otherwise stays on CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}  | num GPUs: {torch.cuda.device_count()}")
N_WORKERS = min(4, os.cpu_count())

Using device: cpu  | num GPUs: 0


# Cloning githubs 

In [ ]:
!git clone https://github.com/mikecheninoulu/SMG

# Loading data 

In [ ]:
#We make sure that the data has the expected format for the model:
DATA = '/kaggle/input/datasets/dandrade05/skeleton-long-seq/experimentWell_skeleton'
folders = sorted(os.listdir(DATA))
print(len(folders), 'folders')
print(folders[:3], '...', folders[-3:])

# inspect ONE folder's contents
one = os.path.join(DATA, folders[0])
print('\ncontents of', folders[0], ':')
for f in sorted(os.listdir(one)):
    print('  ', f)

In [ ]:
!cp "/kaggle/working/SMG/online recognition/SMG/SMGaccessSample.py" .

In [ ]:
"""
Standalone SMG online-recognition feature extraction.
It only uses the loader / feature / HMM-label functions from THDutils.py,
so it imports cleanly on a modern TF2 / PyTorch stack.
Requires SMGaccessSample.py (from the repo) on the path.
"""
from SMGaccessSample import GestureSample

#HELPER FUNCTIONS
def Extract_feature_UNnormalized(smp, used_joints, startFrame, endFrame):
    Skeleton_matrix = np.zeros(shape=(endFrame - startFrame + 1, len(used_joints) * 3))
    frame_num = 0
    for numFrame in range(startFrame, endFrame + 1):
        skel = smp.getSkeleton(numFrame)
        for j in range(len(used_joints)):
            Skeleton_matrix[frame_num, j * 3:(j + 1) * 3] = skel.joins[used_joints[j]][0] #(num_frames, num_joints x3)
        frame_num += 1
    valid_skel = not np.allclose(np.sum(np.abs(Skeleton_matrix)), 0)
    return Skeleton_matrix, valid_skel


def Extract_feature_Realtime(Pose, njoints):
    # Fcc: pairwise joint-coordinate differences within a frame
    FeatureNum = 0
    Fcc = np.zeros(shape=(Pose.shape[0], int(njoints * (njoints - 1) / 2 * 3)))
    for j1 in range(njoints - 1):
        for j2 in range(j1 + 1, njoints):
            Fcc[:, FeatureNum * 3:(FeatureNum + 1) * 3] = \
                Pose[:, j1 * 3:(j1 + 1) * 3] - Pose[:, j2 * 3:(j2 + 1) * 3]
            FeatureNum += 1
    # Fcp: joint-to-joint differences across consecutive frames
    FeatureNum = 0
    Fcp = np.zeros(shape=(Pose.shape[0] - 1, njoints ** 2 * 3))
    for j1 in range(njoints):
        for j2 in range(njoints):
            Fcp[:, FeatureNum * 3:(FeatureNum + 1) * 3] = \
                Pose[1:, j1 * 3:(j1 + 1) * 3] - Pose[0:-1, j2 * 3:(j2 + 1) * 3]
            FeatureNum += 1
    return np.concatenate((Fcc[0:-1, :], Fcp), axis=1)


def extract_HMMstate_label(STATE_NO, action_count, dictionaryNum, gestureID):
    target = np.zeros(shape=(action_count, dictionaryNum))
    for i in range(STATE_NO):
        b = int(np.round(action_count * i / STATE_NO) + 1)
        e = int(np.round(action_count * (i + 1) / STATE_NO))
        seg = e - b + 1
        labels = np.zeros(shape=(dictionaryNum, 1))
        labels[i + STATE_NO * gestureID] = 1
        target[b - 1:e, :] = np.tile(labels.T, (seg, 1))
    return target


def extract_nonActionlabel(action_count, dictionaryNum):
    t = np.zeros(shape=(action_count, dictionaryNum))
    t[:, -1] = 1
    return t


def HMMmatrix(gestureID, frame, frame_count, STATE_NO):
    s1 = int(np.floor(STATE_NO * (frame * 1.0 / (frame_count + 3))) + STATE_NO * gestureID)
    s2 = int(np.floor(STATE_NO * ((frame + 1) * 1.0 / (frame_count + 3))) + STATE_NO * gestureID)
    return s1, s2

In [ ]:
def _process_subject_list(sample_list, parsers, njoints, used_joints, Time_step_NO,
                           featurenum, dictionaryNum, compute_hmm_stats=True):
    """Processes one list of subjects into features/labels, optionally
    accumulating HMM prior/transition stats (only for the training set)."""
    Feature_all_states = np.zeros((400000, featurenum), dtype=np.float32)
    Targets_all_states = np.zeros((400000, dictionaryNum), dtype=np.uint8)
    n_Feature = np.zeros((400000, featurenum), dtype=np.float32)
    n_Targets = np.zeros((400000, dictionaryNum), dtype=np.uint8)
    Prior = np.zeros(dictionaryNum)
    Transition_matrix = np.zeros((dictionaryNum, dictionaryNum))

    cnt = 0
    n_cnt = 0
    for sampleID in sample_list:
        print('\t processing', sampleID)
        smp = GestureSample(os.path.join(parsers['data'], sampleID), sampleID)
        gesturesList = smp.getGestures()
        if compute_hmm_stats:
            Transition_matrix[-1, -1] += smp.getNumFrames()
        MG_flag = smp.getMGFlag()


        for gestureID, startFrame, endFrame in gesturesList:
            SK, _ = Extract_feature_UNnormalized(smp, used_joints, startFrame, endFrame)
            #SK_norm = normalize_skeleton_scale(SK, subj_ref_len)
            F = Extract_feature_Realtime(SK, njoints)
            lab = extract_HMMstate_label(Time_step_NO, F.shape[0], dictionaryNum, gestureID)
            Feature_all_states[cnt:cnt + F.shape[0], :] = F
            Targets_all_states[cnt:cnt + F.shape[0], :] = lab
            cnt += F.shape[0]

            if 1 not in MG_flag[startFrame - 5:startFrame]:
                SK, _ = Extract_feature_UNnormalized(smp, used_joints, startFrame - 5, startFrame)
                #SK_norm = normalize_skeleton_scale(SK, njoints, (4,8))
                F = Extract_feature_Realtime(SK, njoints)
                lab = extract_nonActionlabel(F.shape[0], dictionaryNum)
                n_Feature[n_cnt:n_cnt + F.shape[0], :] = F
                n_Targets[n_cnt:n_cnt + F.shape[0], :] = lab
                n_cnt += F.shape[0]
            if 1 not in MG_flag[endFrame:endFrame + 5]:
                SK, _ = Extract_feature_UNnormalized(smp, used_joints, endFrame, endFrame + 5)
                #SK_norm = normalize_skeleton_scale(SK, njoints, (4,8))
                F = Extract_feature_Realtime(SK, njoints)
                lab = extract_nonActionlabel(F.shape[0], dictionaryNum)
                n_Feature[n_cnt:n_cnt + F.shape[0], :] = F
                n_Targets[n_cnt:n_cnt + F.shape[0], :] = lab
                n_cnt += F.shape[0]

            if compute_hmm_stats:
                for frame in range(endFrame - startFrame + 1 - 4):
                    s1, s2 = HMMmatrix(gestureID, frame, endFrame - startFrame + 1 - 4, Time_step_NO)
                    Prior[s1] += 1
                    Transition_matrix[s1, s2] += 1
                    Transition_matrix[-1, -1] -= 1
                    if frame < 2:
                        Transition_matrix[-1, s1] += 1
                        Prior[-1] += 1
                    if frame > (endFrame - startFrame + 1 - 4 - 2):
                        Transition_matrix[s2, -1] += 1
                        Prior[-1] += 1

    Feature_all_states = Feature_all_states[:cnt, :]
    Targets_all_states = Targets_all_states[:cnt, :]
    n_Feature = n_Feature[:n_cnt, :]
    n_Targets = n_Targets[:n_cnt, :]

    num_samples = min(20000, n_cnt)
    idx = np.random.randint(0, n_cnt, size=num_samples)
    Feature_all = np.concatenate((Feature_all_states, n_Feature[idx]))
    Targets_all = np.concatenate((Targets_all_states, n_Targets[idx]))

    perm = np.random.permutation(Feature_all.shape[0])
    Feature_all, Targets_all = Feature_all[perm], Targets_all[perm]

    return Feature_all, Targets_all, Prior, Transition_matrix


def loader_SMG(parsers, n_train=30):
    t0 = time.time()
    njoints = parsers['njoints']
    used_joints = parsers['used_joints']
    Time_step_NO = parsers['STATE_NUM']
    featurenum = int((njoints * (njoints - 1) / 2 + njoints ** 2) * 3)
    dictionaryNum = parsers['class_count'] * parsers['STATE_NUM'] + 1

    Sample_list = sorted(os.listdir(parsers['data']))
    train_list = Sample_list[0:n_train]
    val_list = Sample_list[n_train:35]

    # training set: computes Prior/Transition stats
    Feature_train, Targets_train, Prior, Transition_matrix = _process_subject_list(
        train_list, parsers, njoints, used_joints, Time_step_NO,
        featurenum, dictionaryNum, compute_hmm_stats=True)

    # validation set: does NOT contribute to Prior/Transition stats
    Feature_val, Targets_val, _, _ = _process_subject_list(
        val_list, parsers, njoints, used_joints, Time_step_NO,
        featurenum, dictionaryNum, compute_hmm_stats=False)

    # fit scaler ONLY on training data, apply to both
    scaler = preprocessing.StandardScaler().fit(Feature_train)
    Feature_train = (Feature_train - scaler.mean_) / scaler.scale_
    Feature_val = (Feature_val - scaler.mean_) / scaler.scale_

    print('done in %d s | train %s val %s'
          % (int(time.time() - t0), Feature_train.shape, Feature_val.shape))

    return Prior, Transition_matrix, Feature_train, Targets_train, Feature_val, Targets_val, scaler

# 1. FEATURE LOADER
In this section we extract and process the input, we create the feature vs target vectors with all the frames + spatial + temporal features.

In [ ]:
parsers = {
    'STATE_NUM': 5,
    'class_count': 17,
    'njoints': 25,
    'data': '/kaggle/input/datasets/dandrade05/skeleton-long-seq/experimentWell_skeleton',  # adjust
    'used_joints': ['SpineBase','SpineMid','ShoulderCenter','Head','ShoulderLeft',
        'ElbowLeft','WristLeft','HandLeft','ShoulderRight','ElbowRight','WristRight',
        'HandRight','HipLeft','KneeLeft','AnkleLeft','FootLeft','HipRight','KneeRight',
        'AnkleRight','FootRight','SpineShoulder','HandTipLeft','ThumbLeft',
        'HandTipRight','ThumbRight'],
}

Prior, Transition_matrix, Feature_train, Targets_train, Feature_val, Targets_val, scaler = loader_SMG(parsers, n_train=30)

print("TRAINING SET")
print('feature dim:', Feature_train.shape[1])          # expect 2275 -> (25·24/2 + 25²)·3 = (300 + 625) ×3
print('num states :', Targets_train.shape[1])          # expect 86  (17*5 + 1)
print('label distribution (argmax):')
import numpy as np
u, c = np.unique(Targets_train.argmax(1), return_counts=True)
freq = dict(zip(u.tolist(), c.tolist()))
print(freq)

print("VALIDATION SET")
print('feature dim:', Feature_val.shape[1])          # expect 2275 -> (25·24/2 + 25²)·3 = (300 + 625) ×3
print('num states :', Targets_val.shape[1])          # expect 86  (17*5 + 1)
print('label distribution (argmax):')
u, c = np.unique(Targets_val.argmax(1), return_counts=True)
print(dict(zip(u.tolist(), c.tolist())))

In [ ]:
print(freq)

In [ ]:
counts_dict = {
    0: 341, 1: 344, 2: 339, 3: 344, 4: 341, 5: 696, 6: 693, 7: 702, 8: 693, 9: 696,
    10: 1193, 11: 1188, 12: 1193, 13: 1188, 14: 1193, 15: 97, 16: 95, 17: 97, 18: 95,
    19: 97, 20: 8336, 21: 8338, 22: 8350, 23: 8338, 24: 8336, 25: 344, 26: 343,
    27: 341, 28: 343, 29: 344, 30: 448, 31: 454, 32: 447, 33: 454, 34: 448,
    35: 2414, 36: 2413, 37: 2412, 38: 2413, 39: 2414, 40: 8559, 41: 8562, 42: 8555,
    43: 8562, 44: 8559, 45: 146, 46: 146, 47: 145, 48: 146, 49: 146, 50: 143,
    51: 144, 52: 143, 53: 144, 54: 143, 55: 2277, 56: 2284, 57: 2281, 58: 2284,
    59: 2277, 60: 691, 61: 692, 62: 691, 63: 692, 64: 691, 65: 778, 66: 774,
    67: 780, 68: 774, 69: 778, 70: 141, 71: 141, 72: 142, 73: 141, 74: 141,
    75: 324, 76: 322, 77: 326, 78: 322, 79: 324, 80: 290, 81: 297, 82: 292,
    83: 297, 84: 290, 85: 18650
}

counts = np.array([freq[i] for i in range(len(freq))], dtype=np.float32)
total_samples = counts.sum()
n_classes = len(counts)

# Square-root smoothed inverse weights
weights_sqrt = np.sqrt(total_samples / counts)
weights_sqrt /= weights_sqrt.mean()  # Scale mean weight to 1.0

class_weights_sqrt = torch.tensor(weights_sqrt, dtype=torch.float32).to(device)
class_weights_sqrt

In [ ]:
print("Features training", Feature_train.shape)
print("Features validation", Feature_val.shape)
print("Priors", Prior.shape)
print("Transition matrix", Transition_matrix.shape)

PRIOR: They're the frame-count (or occurrence-count) of each individual sub-state — recall each gesture is modeled as a short left-to-right chain of state_no HMM sub-states (roughly: onset → early → mid → late → offset). So for gesture 0 you get [131, 126, 125, 126, 85] — close to each other but not identical, because every occurrence of gesture 0 passes through sub-state 0, then sub-state 1, etc., but not every occurrence necessarily spends a countable frame in every sub-state (e.g. very short executions might skip landing squarely on the last sub-state, or the counts come from summed frame durations rather than pure occurrence counts, so they drift downward slightly toward the end of the chain — note gesture 0's last value, 85, drops notably from ~126).

In [ ]:
os.mkdir("/kaggle/working/Online-Recog")

In [ ]:
np.savez("/kaggle/working/Online-Recog/Files.npz", Prior=Prior, 
        Transition_matrix=Transition_matrix,
        Feature_train=Feature_train,
        Targets_train=Targets_train,
        Feature_val=Feature_val,
        Targets_val=Targets_val,
        scaler=scaler)

In [ ]:
def plot_prior(prior, state_no=5, cls_num=17, normalize=True,
               class_names=None, title="Prior distribution over HMM states",
               save_path=None):
    """
    Visualize a (cls_num*state_no + 1)-length prior vector
    Since the prior has no time axis, gesture classes are laid 
    out along x instead of T, each spanning `state_no`
    sub-states, with the background state shown separately at the end.

    prior: 1D array-like of length cls_num*state_no + 1
    """
    prior = np.asarray(prior, dtype=float)
    n_gesture_states = cls_num * state_no
    assert prior.shape[0] == n_gesture_states + 1, \
        f"expected {n_gesture_states + 1} states, got {prior.shape[0]}"

    values = prior / prior.sum() if normalize else prior
    gesture_vals = values[:n_gesture_states].reshape(cls_num, state_no)
    bg_val = values[-1]

    x = np.arange(n_gesture_states)
    cmap = plt.get_cmap('tab20')
    colors = [cmap(k % 20) for k in range(cls_num)]
    bar_colors = np.repeat(colors, state_no, axis=0)

    fig, ax = plt.subplots(figsize=(14, 4))

    # gesture-state bars, colored by class
    ax.bar(x, gesture_vals.flatten(), width=1.0, color=bar_colors,
           edgecolor='white', linewidth=0.3, align='edge')

    # background state, shown as its own bar past the last class,
    # visually separated with a gap + dashed divider
    bg_x = n_gesture_states + 1
    ax.bar([bg_x], [bg_val], width=1.0, color='0.25',
           edgecolor='white', linewidth=0.3, align='edge')
    ax.axvline(n_gesture_states, color='black', linestyle='--',
               linewidth=0.8, alpha=0.6)

    # dashed separators between gesture classes (like the panel gridlines
    # in Fig. 7)
    for k in range(1, cls_num):
        ax.axvline(k * state_no, color='0.7', linestyle='--',
                   linewidth=0.5, zorder=0)

    # x-ticks centered on each class block
    tick_pos = [k * state_no + state_no / 2 for k in range(cls_num)]
    tick_pos.append(bg_x + 0.5)
    labels = class_names if class_names else [f"G{k}" for k in range(cls_num)]
    labels = list(labels) + ["BG"]
    ax.set_xticks(tick_pos)
    ax.set_xticklabels(labels, fontsize=12)

    ax.set_xlim(0, bg_x + 1)
    ax.set_ylabel("Prior probability" if normalize else "Count", fontsize=13)
    ax.set_xlabel("Gesture class (each block = state_no sub-states)", fontsize=13)
    ax.set_title(title, fontsize=14)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    legend_handles = [Patch(facecolor='0.25', label='background state')]
    ax.legend(handles=legend_handles, loc='upper right', fontsize=12, frameon=False)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200)
    plt.show()
    return fig, ax


if __name__ == "__main__":
    # example: your fixed (30-sample) prior vector goes here
    Prior_ = Prior
plot_prior(Prior_, state_no=5, cls_num=17, save_path="/kaggle/working/Online-Recog/Priors.jpeg")

## Understanding our input data and general pipeline
The 2,775 features are coming from the following formula: (njoints·(njoints−1)/2 + njoints²)·3 = (300 + 625)·3 = 925·3 = 2775

Each row of **Feature_all** is one frame, described by a skeleton descriptor.
* Fcc-spatial pose (900 dims). For every unordered pair of joints withing the same frame -> 24*24/2 = 300 pairs, each has (x,y,z) vector -> 900. This encodes the shape of the phose: where joints sit relative to each other right now. It's translation invariant and it's the reason to use pairwise diffs instead of raw coordinates.
* Fcp-motion (1875 dims). For every ordered pair (j1, j2) including j1=j2, the difference between joint j1 at frame t and joint j2 at frame t-1: 25x25 = 625 pairs x 3 = 1875. When j1=j this is that joint's velocity, when j1 is different than j2, it's cross-joint temporal displacement. This block encodes how the body is moving betweenn consecutive frames.

So, 900 (where the body is) + 1875 (how's moving) = 2775 numbers summarizing one frame's pose and dynamics. The 4722 rows simply correspond to the number of frames from those 2 folders (or subjects). Each frame is one training example.

On the other hand,the 86 states we obtained as targets come from:
17 gestures x 5 sub-states + 1 non-gesture (background) state.
The reason of this relies on the nature of the task, we are handling a 15 mins untrimmed stream and asking the model to a) identify where each gesture begins and ends, and b) label it. Trying to use a plain 17-classifier would not be able to do this since it has no notion of "background".

1. Each gesture is modeled as a little left-to-right HMM with 5 states-> begin-> early -> mid -> late -> end. A gesture isn't one label, it's a trajectory through phases, so each gesture owns 5 states. The "no gesture happening" corresponds to the long stretches between MGs and illustrative gesturing.
2. The neural network is then a **per-frame state classifier**: feature vector (2775) -> probability over the 86 states. In HMM terms this is the emission probabilty P(frame|state) "given the model is in this state, how likely is this observed frame?" That's why the target is states, not gestures.
3. A separate **HMM transition matrix** (prior/transition_matrix) encodes which moves are legal: we must pass begin -> mid -> end in order, so we must dip through the background state between gestures.
4. **Viterbi decoding** then fuses the NN's per frame emission scores with the transition rules to find the single most likely path of states through the entire sequence. A continuous run through some gesture's 5 states = one detected instance, and where that run starts/ends gives the best boundaries.

So the NN "maps features into states" because it's the observation model feeding a structured decoder. The 5-states-per-gesture granularity is what lets Viterbi pinpoint boundaries, the background state is what lets it ignore the gaps. The intelligence is split: the NN judges single frames, the HMM+Viterbi imposes temporal structure across the whole stream. 

# 2. EMISSION MODEL
Performs the mapping between the features and the 86 states

In [ ]:
"""
PyTorch emission model for SMG online recognition (DNN-HMM baseline).
Maps a per-frame feature vector (2775-d) -> probability over 86 HMM states
(17 gestures x 5 sub-states + 1 background). 

Inputs come straight from loader_SMG: Feature_all (N, 2775), Targets_all (N, 86)
one-hot. Targets are converted to integer state labels internally.
"""
class EmissionMLP(nn.Module):
    def __init__(self, in_dim, n_states, hidden=1000, p_drop=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),                 
            nn.Dropout(p_drop),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(p_drop),
            nn.BatchNorm1d(hidden),
            nn.Linear(hidden, n_states),
        )

    def forward(self, x):
        return self.net(x)             # raw logits; softmax applied at decode time

In [ ]:
# ----------------------------------------------------------------------
# generic trainer (works for AE_BiLSTM or the MLP) with frame-accuracy check
# ----------------------------------------------------------------------
@torch.no_grad()
def _accuracy(model, loader, device, bg_state, crit=None):
    """Eval-mode pass. Returns (loss, acc, acc_fg). loss is None if crit not given."""
    model.eval()
    correct = total = correct_fg = total_fg = 0
    running = 0.0
    all_preds = []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        if crit is not None:
            running += crit(logits, yb).item() * xb.size(0)
        pred = logits.argmax(1)
        all_preds.append(pred)
        correct += (pred == yb).sum().item(); total += yb.numel()
        fg = yb != bg_state
        correct_fg += (pred[fg] == yb[fg]).sum().item(); total_fg += fg.sum().item()
    loss = running / total if crit is not None else None
    return loss, correct / total, correct_fg / max(total_fg, 1)

import copy

def train_model(model, Feature_train, Targets_train, Feature_val, Targets_val,
                epochs=20, batch_size=32, lr=1e-3, weight_decay=0.0,
                device=None, seed=42, factor=0.2, shuffle_train=True, weights=False):
    # device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    # torch.manual_seed(seed); np.random.seed(seed)
    model = model.to(device)
    n_states = Targets_train.shape[1]
    bg_state = n_states - 1
    y_train = Targets_train.argmax(1).astype(np.int64)
    y_val = Targets_val.argmax(1).astype(np.int64)
    def loader(Features, Targets, shuffle):
        X = torch.tensor(Features, dtype=torch.float32)
        Y = torch.tensor(Targets, dtype=torch.long)
        return DataLoader(TensorDataset(X, Y), batch_size=batch_size, shuffle=shuffle)
    tr = loader(Feature_train, y_train, shuffle_train)
    va = loader(Feature_val, y_val, False)

    # non-shuffled pass over train set, for computing train accuracy/loss post-epoch
    tr_eval = loader(Feature_train, y_train, False)
    optimizer = torch.optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, "min", factor=factor, patience=3)

    if weights:
        crit = nn.CrossEntropyLoss(weight=class_weights_sqrt)
    else:
        crit = nn.CrossEntropyLoss()
        
    print(f'device={device}  params={sum(p.numel() for p in model.parameters())/1e6:.1f}M  '
          f'background frac={(y_train==bg_state).mean():.3f}')
    history = {
        'epoch': [],
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'val_acc_fg': [],
    }
    best = 0.0
    best_epoch = 0
    best_state = None
    for ep in range(1, epochs + 1):
        model.train(); run = 0.0
        for xb, yb in tr:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad(); loss = crit(model(xb), yb); loss.backward(); optimizer.step()
            run += loss.item() * xb.size(0)
        train_loss = run / len(Feature_train)
        _, train_acc, _ = _accuracy(model, tr_eval, device, bg_state, crit=None)
        val_loss, acc, acc_fg = _accuracy(model, va, device, bg_state, crit=crit)

        # checkpoint the weights whenever val accuracy improves
        if acc > best:
            best = acc
            best_epoch = ep
            best_state = copy.deepcopy(model.state_dict())

        #Scheduler to reduce lr
        scheduler.step(val_loss)

        history['epoch'].append(ep)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(acc)
        history['val_acc_fg'].append(acc_fg)
        print(f'epoch {ep:3d} | loss {train_loss:.4f} | val loss {val_loss:.4f} | '
              f'train acc {train_acc:.4f} | val acc {acc:.4f} | '
              f'val acc (gesture states) {acc_fg:.4f}')

    # restore the best-val-accuracy weights before returning
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f'\nrestored weights from epoch {best_epoch} (val acc {best:.4f})')
    print(f'best val frame accuracy: {best:.4f}  (chance {1.0/n_states:.4f})')
    return model, history

def plot_training_history(history, save_path=None):
    """Plot train/val loss and val accuracy curves from the history dict
    returned by train_emission."""
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(history['epoch'], history['train_loss'], label='train loss')
    axes[0].plot(history['epoch'], history['val_loss'], label='val loss')
    axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(history['epoch'], history['val_acc'], label='val acc (all states)')
    axes[1].plot(history['epoch'], history['val_acc_fg'], label='val acc (gesture states only)')
    axes[1].set_xlabel('epoch'); axes[1].set_ylabel('accuracy')
    axes[1].set_title('Validation accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200)
    plt.show()
    return fig, axes


def save_emission(model, path='emission_mlp.pt'):
    torch.save(model.state_dict(), path)
    print('saved ->', path)

In [ ]:
# np.save("/kaggle/working/Online-Recog/history_MLP2.npy", history_MLP2)
# plot_training_history(history_MLP2, save_path="/kaggle/working/Online-Recog/train_plot_MLP_weighted.jpeg")

# 3. NUMPY VITERBI
Here, we run the existing scoring loop to get recall/accuracy/F1

In [ ]:
"""
Stage 3 of the SMG online-recognition pipeline: decode + evaluate.
 
Here we take the trained PyTorch emission model + the HMM Prior/Transition matrices +
the StandardScaler (all from stage 1) and, for each test subject (folders 35:):
  1. extracts full-sequence per-frame features,
  2. runs the emission model -> per-frame state probabilities,
  3. applies the paper's attention emission transform (AES, mu/lambda),
  4. Viterbi-decodes the most likely state path over the whole sequence,
  5. cleans the path into (label, begin, end) segments,
  6. matches segments to ground truth and accumulates recall / precision / F1.
 
Sticks to the original THDutils.tester_SMG
"""

from SMGaccessSample import GestureSample
#from smg_features import Extract_feature_UNnormalized, Extract_feature_Realtime


# ----------------------------------------------------------------------
# emission probabilities from the torch model (+ attention transform)
# ----------------------------------------------------------------------
@torch.no_grad()
def emission_logprob(model, Feature, mean, std, device,
                     aes=True, mu=-2.5, lamd=2.1, eps=1e-8):
    """Feature: (T, 2775) raw. Returns log emission likelihood (n_states, T)."""
    Fn = (Feature - mean) / std
    model.eval()
    logits = model(torch.tensor(Fn, dtype=torch.float32, device=device))
    probs = torch.softmax(logits, dim=1).cpu().numpy()        # (T, n_states)
    y = probs.T  # (n_states, T)
    #print("Sum of log probabilities for one frame: ", y[:, 0].sum())
    
    if aes:
        observ = y.copy()
        atten = y[:-1, :] * y[:-1, :] / 9                      # gesture states only
        SFMX = softmax(atten, axis=0)
        observ[:-1, :] = SFMX * y[:-1, :] * mu + y[:-1, :]
        observ = np.log(np.clip(observ, eps, None))            # eps avoids log(0)
        observ[-1, :] = observ[-1, :] * lamd                   # weight background
    else:
        observ = np.log(np.clip(y, eps, None))
    return observ

@torch.no_grad()
def emission_logprob_bilstm(model, Feature, mean, std, device,
                            aes=True, mu=-2.5, lamd=2.1, eps=1e-8,
                            chunk_size=4096, context=256, n_states = 86):
    """Feature: (T, 2775) raw. Returns log emission likelihood (n_states, T).
    Runs the BiLSTM in overlapping chunks to bound GPU memory, using `context`
    frames of left/right padding per chunk so the bidirectional recurrence
    isn't reset at chunk boundaries; only the non-overlapping center of each
    chunk's output is kept."""
    Fn = (Feature - mean) / std
    model.eval()
    T = Fn.shape[0]
    
    logits_out = torch.empty(T, n_states)
    # safer: infer n_states from first chunk instead of assuming an attribute
    outputs = []
    starts = list(range(0, T, chunk_size))

    for start in starts:
        end = min(start + chunk_size, T)
        ctx_start = max(0, start - context)
        ctx_end = min(T, end + context)

        xb = torch.tensor(Fn[ctx_start:ctx_end], dtype=torch.float32, device=device)
        logits = model(xb) #xb.unsqueeze(0)).squeeze(0)   # add/remove batch dim if your BiLSTM expects (B, T, C)
        logits = logits.cpu()

        # trim off the padding to keep only [start:end)
        left_trim = start - ctx_start
        right_trim = ctx_end - end
        core = logits[left_trim: logits.shape[0] - right_trim if right_trim > 0 else logits.shape[0]]
        outputs.append(core)

        del xb, logits
        torch.cuda.empty_cache()

    logits_full = torch.cat(outputs, dim=0)          # (T, n_states)
    probs = torch.softmax(logits_full, dim=1).numpy()
    y = probs.T                                       # (n_states, T)

    if aes:
        observ = y.copy()
        atten = y[:-1, :] * y[:-1, :] / 9
        SFMX = softmax(atten, axis=0)
        observ[:-1, :] = SFMX * y[:-1, :] * mu + y[:-1, :]
        observ = np.log(np.clip(observ, eps, None))
        observ[-1, :] = observ[-1, :] * lamd
    else:
        observ = np.log(np.clip(y, eps, None))
    return observ


# ----------------------------------------------------------------------
# Viterbi (vectorized; mathematically identical to THDutils.viterbi_path_log)
# ----------------------------------------------------------------------
def viterbi_path_log(prior, transmat, observ_likelihood):
    """prior: (1, N) log-prior. transmat: (N, N) log-transition. obs: (N, T) log."""
    N, T = observ_likelihood.shape
    path = np.zeros(T, dtype=np.int32)
    global_score = np.zeros((N, T))
    back = np.zeros((N, T), dtype=np.int32)
    prior_vec = prior[0]                                       # (N,)
 
    global_score[:, 0] = observ_likelihood[:, 0]
    for t in range(1, T):
        # M[i, j] = global_score[i, t-1] + transmat[i, j] - prior[i] + obs[j, t]
        M = (global_score[:, t - 1][:, None] + transmat
             - prior_vec[:, None] + observ_likelihood[:, t][None, :])
        global_score[:, t] = M.max(axis=0)
        back[:, t] = M.argmax(axis=0)
 
    path[T - 1] = global_score[:, T - 1].argmax()
    for t in range(T - 2, -1, -1):
        path[t] = back[path[t + 1], t + 1]
    return path, back, global_score
 
 
def viterbi_colab_MES(path, global_score, state_no=5, threshold=-3,
                      mini_frame=15, cls_num=17):
    """Clean the Viterbi path into (pred_label, begin, end) gesture segments."""
    all_label = state_no * cls_num
    start_label = np.concatenate((range(0, all_label, state_no),
                                  range(1, all_label, state_no),
                                  range(2, all_label, state_no)))
    end_label = np.concatenate((range(state_no - 3, all_label, state_no),
                                range(state_no - 2, all_label, state_no),
                                range(state_no - 1, all_label, state_no)))
    begin_frame, end_frame, pred_label = [], [], []
    frame = 1
    while frame < path.shape[-1] - 1:
        if path[frame - 1] == all_label and path[frame] in start_label:
            begin_frame.append(frame)
            pred_label.append(int(path[frame] / state_no))
            while frame < path.shape[-1] - 1:
                if path[frame] in end_label and path[frame + 1] == all_label:
                    end_frame.append(frame)
                    break
                frame += 1
        frame += 1
 
    begin_frame = np.array(begin_frame); end_frame = np.array(end_frame)
    pred_label = np.array(pred_label)
    if len(begin_frame) > len(end_frame):
        begin_frame = begin_frame[:-1]; pred_label = pred_label[:-1]
    elif len(begin_frame) < len(end_frame):
        end_frame = end_frame[1:]
 
    flen = end_frame - begin_frame
    mask = flen > mini_frame
    begin_frame, end_frame, pred_label = begin_frame[mask], end_frame[mask], pred_label[mask]
 
    score = []
    for idx, g in enumerate(begin_frame):
        s = global_score[path[end_frame[idx]], end_frame[idx]] - global_score[path[g], g]
        score.append(s)
    score = np.array(score)
    flen = end_frame - begin_frame
    score = score / np.maximum(flen, 1)
    mask = score > threshold
    return pred_label[mask], begin_frame[mask], end_frame[mask]
    

def _build_ground_truth_states(gesturesList, T, Time_step_NO, dictionaryNum, background_state=None):
    """Builds a (T,) array of ground-truth hidden-state indices, one per
    frame, aligned with the time axis of `obs` (n_states, T) returned by
    emission_logprob / emission_logprob_bilstm.

    Any frame not covered by a gesture segment is assigned `background_state`
    (defaults to the last state index, dictionaryNum - 1, matching the
    convention used elsewhere in this pipeline, e.g. Transition_matrix[-1, -1]).

    Parameters
    ----------
    gesturesList : list of (gt_label, gt_b, gt_e)
        Ground-truth gestures for this sample, as returned by smp.getGestures().
    T : int
        Total number of frames in this sample (Feature.shape[0]).
    Time_step_NO, dictionaryNum : as used elsewhere in this pipeline.
    background_state : int, optional
        State index for non-gesture frames. Defaults to dictionaryNum - 1.

    Returns
    -------
    gt_states : np.ndarray, shape (T,), dtype int
        Per-frame ground-truth hidden-state index.
    """
    if background_state is None:
        background_state = dictionaryNum - 1

    gt_states = np.full(T, background_state, dtype=np.int64)

    for gt_label, gt_b, gt_e in gesturesList:
        gt_b_c = max(int(gt_b), 0)
        gt_e_c = min(int(gt_e), T)
        seg_len = gt_e_c - gt_b_c
        if seg_len <= 0:
            continue
        # extract_HMMstate_label returns a one-hot (seg_len, dictionaryNum)
        # array during training-data construction; reuse it here so the
        # ground truth reflects the exact same state-assignment convention used for training.
        lab = extract_HMMstate_label(Time_step_NO, seg_len, dictionaryNum, gt_label)
        gt_states[gt_b_c:gt_e_c] = np.argmax(lab, axis=1)

    return gt_states


def evaluate_online(model, parsers, scaler, Prior, Transition_matrix,
                     n_test_start=35, threshold_alpha=0.3, mu=-2.5, lamd=2.1,
                     device=None, eps=1e-8, verbose=True, bilstm=False,
                     n_gt_samples=5):
    """Same as the original evaluate_online, with one addition: builds
    per-frame ground-truth hidden-state arrays (aligned with each sample's
    `obs` emission matrix) for the first `n_gt_samples` test samples, and
    returns them as `ground_truth_states` (a list of length <= n_gt_samples,
    each element a (T,) int array).

    This will allow us you compare, e.g. via plot_posteriorgram(obs, true_states=gt),
    what the model predicted against what it should have predicted, BEFORE
    the HMM/Viterbi decoding step smooths/reshapes things.
    """
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    mean, std = scaler.mean_, scaler.scale_
    used_joints, njoints = parsers['used_joints'], parsers['njoints']
    state_no, cls_num = parsers['STATE_NUM'], parsers['class_count']
    dictionaryNum = Transition_matrix.shape[0]  # n_states, incl. background

    log_prior = np.log(np.clip(Prior, eps, None)).reshape(1, -1)   # (1, N)
    log_trans = np.log(np.clip(Transition_matrix, eps, None))      # (N, N)

    Sample_list = sorted(os.listdir(parsers['data']))
    correct, recall_len, pred_len_tot = 0.0, 0, 0
    t0 = time.time()
    predictions = []
    targets = []
    hidden_states = []
    ground_truth_states = []   # list of length <= n_gt_samples
    total_frames = 0
    for sampleID in Sample_list[n_test_start:]:
        ts = time.time()
        smp = GestureSample(os.path.join(parsers['data'], sampleID), sampleID)
        gesturesList = smp.getGestures()
        frame_count = smp.getNumFrames()

        SK, _ = Extract_feature_UNnormalized(smp, used_joints, 1, frame_count - 1)
        Feature = Extract_feature_Realtime(SK, njoints)
        total_frames += Feature.shape[0]
        if bilstm == False:
            obs = emission_logprob(model, Feature, mean, std, device, True, mu, lamd, eps)
        else:
            obs = emission_logprob_bilstm(model, Feature, mean, std, device,
                            aes=True, mu=-2.5, lamd=2.1, eps=1e-8,
                            chunk_size=4096, context=500)

        obs[-1, 0:5] = 0; obs[-1, -5:] = 0          # force background at the edges

        # build frame-level ground truth for this sample, capped at
        # n_gt_samples total across the loop (not the first 5 subjects in
        # Sample_list, but the first 5 that actually get processed here).
        if len(ground_truth_states) < n_gt_samples:
            gt_states = _build_ground_truth_states(
                gesturesList, obs.shape[1], Time_step_NO=state_no,
                dictionaryNum=dictionaryNum,
            )
            ground_truth_states.append(gt_states)

        path, _, gscore = viterbi_path_log(log_prior, log_trans, obs)
        pred_label, begin_frame, end_frame = viterbi_colab_MES(
            path, gscore, state_no=state_no, threshold=-3, mini_frame=15, cls_num=cls_num)
        preds = (sampleID, pred_label, begin_frame, end_frame)
        predictions.append(preds)
        targets.append(gesturesList)
        hidden_states.append(obs)

        # match predictions to ground truth (same label + temporal IoU > alpha)
        cur_correct = 0.0
        for gt_label, gt_b, gt_e in gesturesList:
            for i in range(len(begin_frame)):
                if pred_label[i] == gt_label:
                    inter = min(end_frame[i], gt_e) - max(begin_frame[i], gt_b)
                    union = max(end_frame[i], gt_e) - min(begin_frame[i], gt_b)
                    if union > 0 and inter / union > threshold_alpha:
                        correct += 1; cur_correct += 1
                        break

        recall_len += len(gesturesList)
        pred_len_tot += len(pred_label)
        if verbose:
            r = cur_correct / max(len(gesturesList), 1)
            p = cur_correct / max(len(pred_label), 1)
            f1 = 2 * r * p / (r + p + 1e-7)
            print(f'{sampleID}: GT={len(gesturesList)} pred={len(pred_label)} '
                  f'F1={f1:.3f} R={r:.3f} P={p:.3f}  ({time.time()-ts:.0f}s)')

    print("Total frames: ", total_frames)
    recall = correct / max(recall_len, 1)
    precision = correct / max(pred_len_tot, 1)
    f1 = 2 * recall * precision / (recall + precision + 1e-7)
    print('\n==== online recognition (test subjects) ====')
    print(f'recall    = {recall:.4f}')
    print(f'precision = {precision:.4f}')
    print(f'F1 score  = {f1:.4f}')
    print(f'(paper AED-BiLSTM ~ R 0.245 / acc 0.164 / F1 0.203)   {time.time()-t0:.0f}s')
    return (dict(recall=recall, precision=precision, f1=f1),
            predictions, targets, hidden_states, ground_truth_states)

In [ ]:
# #from smg_decode import evaluate_online

# results_MLP, predictions_MLP, targets, obs_states, true_states = evaluate_online(model_MLP, parsers, scaler, Prior, Transition_matrix)
# np.save("observ_MLP.npy", obs_states[0])  

In [ ]:
# np.savez("/kaggle/working/Online-Recog/preds_MLP.npz", 
#          results=results_MLP, predictions=np.array(predictions_MLP, dtype=object), targets=np.array(targets, dtype=object)
# ) 

In [ ]:
# #Weighted Cross Entropy Results
# results_MLP2, predictions_MLP2, targets, obs_states2, true_states = evaluate_online(model_MLP2, parsers, scaler, Prior, Transition_matrix)
# np.save("observ_MLP2.npy", obs_states2[0])  

In [ ]:
# np.savez("/kaggle/working/Online-Recog/preds_MLP2.npz", 
#          results=results_MLP2, predictions=np.array(predictions_MLP2, dtype=object), targets=np.array(targets, dtype=object)
# ) 

In [ ]:
"""
Posteriorgram plotting utility for HMM emission log-probability matrices.

Expects a matrix `observ` of shape (n_states, T), where
    observ[i, t] = log P(obs_t | state_i)
(i.e. the exact output of your `emission_logprob` function).

Usage:
    from plot_posteriorgram import plot_posteriorgram
    plot_posteriorgram(observ, save_path="posteriorgram.png")
"""

def plot_posteriorgram(
    observ,
    state_labels=None,
    true_states=None,
    zoom=None,
    title="Emission log-probability posteriorgram",
    save_path=None,
    figsize=(16, 6),
    cmap="viridis",
):
    """
    Plot a heatmap of an (n_states, T) log-probability / emission matrix,
    with time on the x-axis and hidden state on the y-axis.

    Parameters
    ----------
    observ : np.ndarray, shape (n_states, T)
        Log-probability (or any real-valued score) matrix. This is exactly
        what `emission_logprob(...)` returns.
    state_labels : list[str], optional
        Length n_states. Used as y-axis tick labels. Defaults to 0..n_states-1.
    true_states : array-like, shape (T,), optional
        Ground-truth state index per frame. If given, overlaid as a red line
        on top of the heatmap and plotted argmax comparison below it.
    zoom : tuple(int, int), optional
        (start, end) frame range for a second, zoomed-in panel. If None,
        only the full-sequence view is plotted.
    title : str
        Plot title.
    save_path : str, optional
        If given, the figure is saved to this path (png).
    figsize : tuple
        Figure size for the full view. The zoomed panel reuses the same width.
    cmap : str
        Matplotlib colormap name.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    n_states, T = observ.shape
    argmax_states = np.argmax(observ, axis=0)  # predicted state per frame

    n_panels = 2 if zoom is not None else 1
    fig, axes = plt.subplots(
        n_panels, 1, figsize=(figsize[0], figsize[1] * n_panels), squeeze=False
    )
    axes = axes[:, 0]

    def _draw(ax, t0, t1, panel_title):
        sub = observ[:, t0:t1]
        im = ax.imshow(
            sub,
            aspect="auto",
            origin="lower",
            cmap=cmap,
            extent=[t0, t1, -0.5, n_states - 0.5],
            interpolation="nearest",
        )
        ax.set_xlabel("Frame (time)")
        ax.set_ylabel("Hidden state")
        ax.set_title(panel_title)

        if state_labels is not None:
            ax.set_yticks(range(n_states))
            ax.set_yticklabels(state_labels, fontsize=7)

        # overlay predicted state (argmax) as a thin white line
        ax.plot(
            np.arange(t0, t1),
            argmax_states[t0:t1],
            color="white",
            linewidth=0.7,
            alpha=0.8,
            label="argmax (predicted)",
        )

        # overlay ground truth if provided
        if true_states is not None:
            ax.plot(
                np.arange(t0, t1),
                np.asarray(true_states)[t0:t1],
                color="red",
                linewidth=0.9,
                alpha=0.9,
                linestyle="--",
                label="ground truth",
            )
            ax.legend(loc="upper right", fontsize=7)

        fig.colorbar(im, ax=ax, label="log P(obs | state)")

    _draw(axes[0], 0, T, f"{title} — full sequence (T={T})")

    if zoom is not None:
        z0, z1 = zoom
        _draw(axes[1], z0, z1, f"{title} — zoom [{z0}:{z1}]")

    fig.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")

    return fig

# if __name__ == "__main__":
#     # quick self-test on a random example
#     import sys

#     #path = sys.argv[1] if len(sys.argv) > 1 else "observ.npy"
#     observ = obs_states[0]
#     fig = plot_posteriorgram(
#         observ, true_states= true_states[0],
#         zoom=(0, min(1500, observ.shape[1])),
#         save_path="posteriorgram.png",
#     )
#     print("Saved posteriorgram.png")

In [ ]:
# #Heat-map MLP
# if __name__ == "__main__":
#     # quick self-test on a random example
#     import sys

#     #path = sys.argv[1] if len(sys.argv) > 1 else "observ.npy"
#     observ = obs_states[0]
#     fig = plot_posteriorgram(
#         observ, true_states= true_states[0],
#         zoom=(0, min(1500, observ.shape[1])),
#         save_path="posteriorgram_MLP.png",
#     )
#     print("Saved posteriorgram.png")

In [ ]:
# #Heat-map weighted MLP
# if __name__ == "__main__":
#     # quick self-test on a random example
#     import sys

#     #path = sys.argv[1] if len(sys.argv) > 1 else "observ.npy"
#     observ = obs_states2[0]
#     fig = plot_posteriorgram(
#         observ, true_states= true_states[0],
#         zoom=(0, min(1500, observ.shape[1])),
#         save_path="posteriorgram_MLP2.png",
#     )
#     print("Saved posteriorgram.png")

In [ ]:
"""
Plots predicted gesture segments against ground-truth gesture segments over
time, handling their different input layouts:

- predictions: array of shape (3, N_pred) -- three PARALLEL ROWS:
    predictions[0] = gesture IDs
    predictions[1] = begin frames
    predictions[2] = end frames
  (this is the (pred_label, begin_frame, end_frame) triple from
   viterbi_colab_MES / evaluate_online, stacked into one array)

- targets: array/list of shape (N_gt, 3) -- one ROW per gesture:
    targets[i] = [gesture_id, begin_frame, end_frame]
  (this is gesturesList, i.e. smp.getGestures())

Both get converted to the same (begin, end, label) segment representation,
then drawn as step-function-style horizontal segments: x-axis = time
(frame), y-axis = gesture/state ID, flat at `label` for [begin, end) and
implicitly at a "background" level everywhere else.
"""
import numpy as np
import matplotlib.pyplot as plt


def _normalize_segments(data, layout):
    """Returns a list of (label, begin, end) tuples regardless of input layout.

    layout : {"cols", "rows"}
        "rows" -- shape (N, 3), one gesture per row: [label, begin, end]
        "cols" -- shape (3, N), one gesture per COLUMN, rows are
                  [labels, begins, ends] (the predictions format)
    """
    arr = np.asarray(data)
    if layout == "rows":
        # (N, 3): iterate rows directly
        return [(int(r[0]), int(r[1]), int(r[2])) for r in arr]
    elif layout == "cols":
        # (3, N): rows are label/begin/end -- transpose first
        labels, begins, ends = arr[0], arr[1], arr[2]
        return [(int(l), int(b), int(e)) for l, b, e in zip(labels, begins, ends)]
    else:
        raise ValueError(f"unknown layout: {layout}")


def plot_pred_vs_gt(predictions, targets, background_label=None,
                     title="Predicted vs. ground-truth gesture segments",
                     figsize=(16, 5), save_path=None):
    """
    Parameters
    ----------
    predictions : array-like, shape (3, N_pred)
        Rows = [gesture_id, begin_frame, end_frame], as produced by
        viterbi_colab_MES / evaluate_online's `preds` tuple.
    targets : array-like, shape (N_gt, 3)
        Rows = [gesture_id, begin_frame, end_frame], as returned by
        smp.getGestures().
    background_label : int, optional
        Y-value to draw between segments (i.e. when nothing is active).
        Defaults to -1 (visually separated below the lowest real label)
        if not given.
    title : str
        Plot title.
    figsize : tuple
    save_path : str, optional
        If given, saves the figure as a PNG.

    Returns
    -------
    fig : matplotlib.figure.Figure
    """
    pred_segs = _normalize_segments(predictions, layout="cols")
    gt_segs = _normalize_segments(targets, layout="rows")

    all_labels = [s[0] for s in pred_segs] + [s[0] for s in gt_segs]
    if background_label is None:
        background_label = (min(all_labels) - 1) if all_labels else -1

    t_max = max(
        [s[2] for s in pred_segs] + [s[2] for s in gt_segs] + [0]
    )

    def _to_step_arrays(segs, t_max, background_label):
        """Builds (x, y) step-plot arrays: flat at `label` during [begin,end),
        dropping to `background_label` in the gaps between segments."""
        x, y = [0], [background_label]
        for label, b, e in sorted(segs, key=lambda s: s[1]):
            x += [b, b, e, e]
            y += [background_label, label, label, background_label]
        x += [t_max]
        y += [background_label]
        return x, y

    pred_x, pred_y = _to_step_arrays(pred_segs, t_max, background_label)
    gt_x, gt_y = _to_step_arrays(gt_segs, t_max, background_label)

    fig, axes = plt.subplots(2, 1, figsize=figsize, sharex=True)

    axes[0].step(gt_x, gt_y, where="post", color="tab:green", linewidth=1.2)
    axes[0].set_ylabel("Gesture ID")
    axes[0].set_title(f"{title} — ground truth ({len(gt_segs)} segments)")
    axes[0].grid(alpha=0.3)

    axes[1].step(pred_x, pred_y, where="post", color="tab:orange", linewidth=1.2)
    axes[1].set_ylabel("Gesture ID")
    axes[1].set_xlabel("Frame (time)")
    axes[1].set_title(f"predicted ({len(pred_segs)} segments)")
    axes[1].grid(alpha=0.3)

    fig.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")

    return fig


def plot_pred_vs_gt_overlay(predictions, targets, background_label=None,
                             title="Predicted vs. ground-truth gesture segments",
                             figsize=(16, 5), save_path=None):
    """Same data, but both series overlaid on a single axis instead of two
    stacked panels -- easier for eyeballing exact frame-level alignment
    over a shorter time range (e.g. after zooming in)."""
    pred_segs = _normalize_segments(predictions, layout="cols")
    gt_segs = _normalize_segments(targets, layout="rows")

    all_labels = [s[0] for s in pred_segs] + [s[0] for s in gt_segs]
    if background_label is None:
        background_label = (min(all_labels) - 1) if all_labels else -1

    t_max = max([s[2] for s in pred_segs] + [s[2] for s in gt_segs] + [0])

    def _to_step_arrays(segs, t_max, background_label):
        x, y = [0], [background_label]
        for label, b, e in sorted(segs, key=lambda s: s[1]):
            x += [b, b, e, e]
            y += [background_label, label, label, background_label]
        x += [t_max]
        y += [background_label]
        return x, y

    pred_x, pred_y = _to_step_arrays(pred_segs, t_max, background_label)
    gt_x, gt_y = _to_step_arrays(gt_segs, t_max, background_label)

    fig, ax = plt.subplots(figsize=figsize)
    ax.step(gt_x, gt_y, where="post", color="tab:green", linewidth=1.6,
            label=f"ground truth ({len(gt_segs)})", alpha=0.8)
    ax.step(pred_x, pred_y, where="post", color="tab:orange", linewidth=1.2,
            label=f"predicted ({len(pred_segs)})", alpha=0.8, linestyle="--")
    ax.set_xlabel("Frame (time)")
    ax.set_ylabel("Gesture ID")
    ax.set_title(title)
    ax.legend(loc="upper right")
    ax.grid(alpha=0.3)
    fig.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")

    return fig

In [ ]:
# preds_MLP = predictions_MLP[0][1:4]
# np.save("predictions_MLP.npy", preds_MLP) 
# np.save("targets.npy", targets[0]) 

# if __name__ == "__main__":
#     preds_MLP = np.load("predictions_MLP.npy", allow_pickle=True)
#     targets_ = np.load("targets.npy", allow_pickle=True)

#     fig1 = plot_pred_vs_gt(preds_MLP, targets_, save_path="pred_vs_gt_stacked_MLP.png")
#     fig2 = plot_pred_vs_gt_overlay(preds_MLP, targets_, save_path="pred_vs_gt_overlay_MLP.png")
#     print("Saved pred_vs_gt_stacked.png and pred_vs_gt_overlay.png")

In [ ]:
# preds_MLP2 = predictions_MLP2[0][1:4]
# np.save("predictions_MLP2.npy", preds_MLP2) 

# if __name__ == "__main__":
#     preds_MLP2 = np.load("predictions_MLP2.npy", allow_pickle=True)
#     # targets = np.load("targets_bilstm.npy", allow_pickle=True)
   
#     fig1 = plot_pred_vs_gt(preds_MLP2, targets_, save_path="pred_vs_gt_stacked_MLP2_weighted.png")
#     fig2 = plot_pred_vs_gt_overlay(preds_MLP2, targets_, save_path="pred_vs_gt_overlay_MLP2_weighted.png")
#     print("Saved pred_vs_gt_stacked.png and pred_vs_gt_overlay.png")

# Family of RNNs: AED-BiLSTM and variants

In [ ]:
"""
AE-BiLSTM emission model for SMG online recognition:

    reshape (2775) -> (timesteps=5, 555)
    BiGRU(units*2) , return_sequences
    temporal attention   (attention_3d_block)
    BiGRU(units)    , return_sequences
    spatial attention    (attention_spatial_block)
    Dense(1000, relu) -> Dropout(0.3) -> BatchNorm -> Flatten
    Dense(n_states)   [raw logits; decode applies softmax]
"""
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


class TemporalAttention(nn.Module):
    """attention_3d_block: per-channel softmax weighting across timesteps."""
    def __init__(self, time_steps):
        super().__init__()
        self.dense = nn.Linear(time_steps, time_steps)

    def forward(self, x):                  # x: (B, T, D)
        a = x.transpose(1, 2)              # (B, D, T)
        a = torch.softmax(self.dense(a), dim=-1)   # softmax over T
        a = a.transpose(1, 2)              # (B, T, D)
        return x * a


class SpatialAttention(nn.Module):
    """attention_spatial_block: per-timestep softmax weighting across features."""
    def __init__(self, input_dim):
        super().__init__()
        self.dense = nn.Linear(input_dim, input_dim)

    def forward(self, x):                  # x: (B, T, D)
        a = torch.softmax(self.dense(x), dim=-1)
        return x * a


class AE_BiLSTM(nn.Module):
    def __init__(self, in_dim=2775, n_states=86, timesteps=5, units=500,
                 dense_units=500, p_drop=0.3):
        super().__init__()
        assert in_dim % timesteps == 0, f'{in_dim} not divisible by {timesteps}'
        self.timesteps = timesteps
        self.featurelen = in_dim // timesteps          # 555

        self.gru1 = nn.GRU(self.featurelen, units * 2, batch_first=True,
                           bidirectional=True)          # out: units*4
        self.t_att = TemporalAttention(timesteps)
        self.drop2 = nn.Dropout(p_drop)

        
        self.gru2 = nn.GRU(units * 4, units, batch_first=True,
                           bidirectional=True)           # out: units*2
        self.s_att = SpatialAttention(units * 2)
        self.drop4 = nn.Dropout(p_drop)
        self.dense = nn.Linear(units * 2, dense_units)
        self.drop = nn.Dropout(p_drop)
        self.bn = nn.BatchNorm1d(dense_units)
        self.out = nn.Linear(timesteps * dense_units, n_states)

    def forward(self, x):                  # x: (B, 2775)
        B = x.size(0)
        x = x.view(B, self.timesteps, self.featurelen)
        x, _ = self.gru1(x)                # (B, T, units*4)
        x = self.t_att(x)
        x = self.drop2(x)
        x, _ = self.gru2(x)                # (B, T, units*2)
        x = self.s_att(x)
        x = self.drop4(x)
        x = torch.relu(self.dense(x))      # (B, T, dense_units)
        x = self.drop(x)
        # Keras BatchNormalization(axis=-1) over the feature dim:
        x = self.bn(x.reshape(B * self.timesteps, -1)).reshape(B, self.timesteps, -1)
        x = x.reshape(B, -1)               # flatten
        return self.out(x)                 # logits (B, n_states)

In [ ]:
model_bilstm= AE_BiLSTM()
model_bilstm

## Training 

In [ ]:
model_bilstm, history_bilstm = train_model(model_bilstm, Feature_train, Targets_train, Feature_val, Targets_val,
                epochs=15, batch_size=32, lr=1e-3, weight_decay=1e-4, device=device, seed=42, factor=0.5, weights=False)

In [ ]:
results_bilstm, predictions, targets, obs_states, true_states = evaluate_online(model_bilstm, parsers, scaler, Prior, Transition_matrix, bilstm=True)
save_emission(model_bilstm, '/kaggle/working/Online-Recog/emission_bilstm.pt')

In [ ]:
np.savez("/kaggle/working/Online-Recog/preds_bilstm.npz", 
         results=results_bilstm, predictions=np.array(predictions, dtype=object), targets=np.array(targets, dtype=object)
) 

In [ ]:
if __name__ == "__main__":
    # quick self-test on a random example
    import sys

    #path = sys.argv[1] if len(sys.argv) > 1 else "observ.npy"
    observ = obs_states[0]
    fig = plot_posteriorgram(
        observ, true_states= true_states[0],
        zoom=(0, min(1500, observ.shape[1])),
        save_path="posteriorgram_BiLSTM.png",
    )
    print("Saved posteriorgram.png")

In [ ]:
obs_preds = predictions[0][1:4]
np.save("predictions_bilstm.npy", obs_preds) 
np.save("targets_bilstm.npy", targets[0]) 

if __name__ == "__main__":
    predictions_ = np.load("predictions_bilstm.npy", allow_pickle=True)
    targets_ = np.load("targets_bilstm.npy", allow_pickle=True)

    fig1 = plot_pred_vs_gt(predictions_, targets_, save_path="pred_vs_gt_stacked_bilstm.png")
    fig2 = plot_pred_vs_gt_overlay(predictions_, targets_, save_path="pred_vs_gt_overlay_bilstm.png")
    print("Saved pred_vs_gt_stacked.png and pred_vs_gt_overlay.png")

In [ ]:
model_bilstm, history_bilstm = train_model(model_bilstm, Feature_train, Targets_train, Feature_val, Targets_val,
                epochs=15, batch_size=32, lr=1e-3, weight_decay=1e-4, device=device, seed=42, factor=0.5, weights=True)

In [ ]:
results_bilstm, predictions, targets, obs_states, true_states = evaluate_online(model_bilstm, parsers, scaler, Prior, Transition_matrix, bilstm=True)
save_emission(model_bilstm, '/kaggle/working/Online-Recog/emission_bilstm.pt')

In [ ]:
np.savez("/kaggle/working/Online-Recog/preds_bilstm_weighted.npz", 
         results=results_bilstm, predictions=np.array(predictions, dtype=object), targets=np.array(targets, dtype=object)
) 

In [ ]:
if __name__ == "__main__":
    # quick self-test on a random example
    import sys

    #path = sys.argv[1] if len(sys.argv) > 1 else "observ.npy"
    observ = obs_states[0]
    fig = plot_posteriorgram(
        observ, true_states= true_states[0],
        zoom=(0, min(1500, observ.shape[1])),
        save_path="posteriorgram_BiLSTM_weighted.png",
    )
    print("Saved posteriorgram.png")

In [ ]:
obs_preds = predictions[0][1:4]
np.save("predictions_bilstm_weighted.npy", obs_preds) 
np.save("targets_bilstm_weighted.npy", targets[0]) 

if __name__ == "__main__":
    predictions_ = np.load("predictions_bilstm_weighted.npy", allow_pickle=True)
    targets_ = np.load("targets_bilstm_weighted.npy", allow_pickle=True)

    fig1 = plot_pred_vs_gt(predictions_, targets_, save_path="pred_vs_gt_stacked_bilstm_weighted.png")
    fig2 = plot_pred_vs_gt_overlay(predictions_, targets_, save_path="pred_vs_gt_overlay_bilstm_weighted.png")
    print("Saved pred_vs_gt_stacked.png and pred_vs_gt_overlay.png")

# Ablation models

In [ ]:
"""
Ablation model variants for the AE_BiLSTM architecture.

Progression, from simplest to full model:
    1. SingleGRU            - unidirectional GRU, no attention        (floor baseline)
    2. SingleBiGRU           - bidirectional GRU, no attention         (bidirectionality effect)
    3. BiGRU_TemporalAttOnly - BiGRU + temporal attention only         (t_att contribution)
    4. BiGRU_SpatialAttOnly  - BiGRU + spatial attention only          (s_att contribution)
    5. StackedBiGRU_NoAtt    - two BiGRUs, no attention                (2nd GRU layer effect)
    6. AE_BiLSTM (full)      - GRU -> t_att -> GRU -> s_att            (current architecture)
    7. AE_BiLSTM_Swapped     - GRU -> s_att -> GRU -> t_att            (attention order effect)

"""

# ---------------------------------------------------------------------------
# Attention blocks (unchanged from the original architecture)
# ---------------------------------------------------------------------------

class TemporalAttention(nn.Module):
    """attention_3d_block: per-channel softmax weighting across timesteps."""
    def __init__(self, time_steps):
        super().__init__()
        self.dense = nn.Linear(time_steps, time_steps)

    def forward(self, x):                  # x: (B, T, D)
        a = x.transpose(1, 2)              # (B, D, T)
        a = torch.softmax(self.dense(a), dim=-1)   # softmax over T
        a = a.transpose(1, 2)              # (B, T, D)
        return x * a


class SpatialAttention(nn.Module):
    """attention_spatial_block: per-timestep softmax weighting across features."""
    def __init__(self, input_dim):
        super().__init__()
        self.dense = nn.Linear(input_dim, input_dim)

    def forward(self, x):                  # x: (B, T, D)
        a = torch.softmax(self.dense(x), dim=-1)
        return x * a


# ---------------------------------------------------------------------------
# Shared helper: final classification head
# ---------------------------------------------------------------------------

class ClassifierHead(nn.Module):
    """dense -> dropout -> batchnorm -> flatten -> output, reused by every variant."""
    def __init__(self, in_dim, timesteps, dense_units, n_states, p_drop):
        super().__init__()
        self.timesteps = timesteps
        self.dense = nn.Linear(in_dim, dense_units)
        self.drop = nn.Dropout(p_drop)
        self.bn = nn.BatchNorm1d(dense_units)
        self.out = nn.Linear(timesteps * dense_units, n_states)

    def forward(self, x):                  # x: (B, T, D_in)
        B = x.size(0)
        x = torch.relu(self.dense(x))       # (B, T, dense_units)
        x = self.drop(x)
        x = self.bn(x.reshape(B * self.timesteps, -1)).reshape(B, self.timesteps, -1)
        x = x.reshape(B, -1)                # flatten
        return self.out(x)                  # logits (B, n_states)


def _reshape_input(x, timesteps, featurelen):
    B = x.size(0)
    return x.view(B, timesteps, featurelen)


# ---------------------------------------------------------------------------
# 1. Single unidirectional GRU, no attention (floor baseline)
# ---------------------------------------------------------------------------

class SingleGRU(nn.Module):
    def __init__(self, in_dim=2775, n_states=86, timesteps=5, units=250,
                 dense_units=250, p_drop=0.5):
        super().__init__()
        assert in_dim % timesteps == 0, f'{in_dim} not divisible by {timesteps}'
        self.timesteps = timesteps
        self.featurelen = in_dim // timesteps
        self.gru = nn.GRU(self.featurelen, units, batch_first=True,
                           bidirectional=False)          # out: units
        self.drop = nn.Dropout(p_drop)
        self.head = ClassifierHead(units, timesteps, dense_units, n_states, p_drop)

    def forward(self, x):
        x = _reshape_input(x, self.timesteps, self.featurelen)
        x, _ = self.gru(x)                  # (B, T, units)
        x = self.drop(x)
        return self.head(x)


# ---------------------------------------------------------------------------
# 2. Single bidirectional GRU, no attention
# ---------------------------------------------------------------------------

class SingleBiGRU(nn.Module):
    def __init__(self, in_dim=2775, n_states=86, timesteps=5, units=250,
                 dense_units=250, p_drop=0.5):
        super().__init__()
        assert in_dim % timesteps == 0, f'{in_dim} not divisible by {timesteps}'
        self.timesteps = timesteps
        self.featurelen = in_dim // timesteps
        self.gru = nn.GRU(self.featurelen, units, batch_first=True,
                           bidirectional=True)           # out: units*2
        self.drop = nn.Dropout(p_drop)
        self.head = ClassifierHead(units * 2, timesteps, dense_units, n_states, p_drop)

    def forward(self, x):
        x = _reshape_input(x, self.timesteps, self.featurelen)
        x, _ = self.gru(x)                  # (B, T, units*2)
        x = self.drop(x)
        return self.head(x)


# ---------------------------------------------------------------------------
# 3. BiGRU + temporal attention only
# ---------------------------------------------------------------------------

class BiGRU_TemporalAttOnly(nn.Module):
    def __init__(self, in_dim=2775, n_states=86, timesteps=5, units=250,
                 dense_units=250, p_drop=0.5):
        super().__init__()
        assert in_dim % timesteps == 0, f'{in_dim} not divisible by {timesteps}'
        self.timesteps = timesteps
        self.featurelen = in_dim // timesteps
        self.gru = nn.GRU(self.featurelen, units, batch_first=True,
                           bidirectional=True)           # out: units*2
        self.drop1 = nn.Dropout(p_drop)
        self.t_att = TemporalAttention(timesteps)
        self.drop2 = nn.Dropout(p_drop)
        self.head = ClassifierHead(units * 2, timesteps, dense_units, n_states, p_drop)

    def forward(self, x):
        x = _reshape_input(x, self.timesteps, self.featurelen)
        x, _ = self.gru(x)                  # (B, T, units*2)
        x = self.drop1(x)
        x = self.t_att(x)
        x = self.drop2(x)
        return self.head(x)


# ---------------------------------------------------------------------------
# 4. BiGRU + spatial attention only
# ---------------------------------------------------------------------------

class BiGRU_SpatialAttOnly(nn.Module):
    def __init__(self, in_dim=2775, n_states=86, timesteps=5, units=250,
                 dense_units=250, p_drop=0.5):
        super().__init__()
        assert in_dim % timesteps == 0, f'{in_dim} not divisible by {timesteps}'
        self.timesteps = timesteps
        self.featurelen = in_dim // timesteps
        self.gru = nn.GRU(self.featurelen, units, batch_first=True,
                           bidirectional=True)           # out: units*2
        self.drop1 = nn.Dropout(p_drop)
        self.s_att = SpatialAttention(units * 2)
        self.drop2 = nn.Dropout(p_drop)
        self.head = ClassifierHead(units * 2, timesteps, dense_units, n_states, p_drop)

    def forward(self, x):
        x = _reshape_input(x, self.timesteps, self.featurelen)
        x, _ = self.gru(x)                  # (B, T, units*2)
        x = self.drop1(x)
        x = self.s_att(x)
        x = self.drop2(x)
        return self.head(x)


# ---------------------------------------------------------------------------
# 5. Two stacked BiGRUs, no attention
# ---------------------------------------------------------------------------

class StackedBiGRU_NoAtt(nn.Module):
    def __init__(self, in_dim=2775, n_states=86, timesteps=5, units=250,
                 dense_units=250, p_drop=0.5):
        super().__init__()
        assert in_dim % timesteps == 0, f'{in_dim} not divisible by {timesteps}'
        self.timesteps = timesteps
        self.featurelen = in_dim // timesteps
        self.gru1 = nn.GRU(self.featurelen, units * 2, batch_first=True,
                            bidirectional=True)          # out: units*4
        self.drop1 = nn.Dropout(p_drop)
        self.gru2 = nn.GRU(units * 4, units, batch_first=True,
                            bidirectional=True)          # out: units*2
        self.drop2 = nn.Dropout(p_drop)
        self.head = ClassifierHead(units * 2, timesteps, dense_units, n_states, p_drop)

    def forward(self, x):
        x = _reshape_input(x, self.timesteps, self.featurelen)
        x, _ = self.gru1(x)                 # (B, T, units*4)
        x = self.drop1(x)
        x, _ = self.gru2(x)                 # (B, T, units*2)
        x = self.drop2(x)
        return self.head(x)


# ---------------------------------------------------------------------------
# 6. Full model: GRU -> temporal attention -> GRU -> spatial attention
#    (this is your current architecture, included here for a single
#    consistent reference point across all ablation runs)
# ---------------------------------------------------------------------------

class AE_BiLSTM(nn.Module):
    def __init__(self, in_dim=2775, n_states=86, timesteps=5, units=250,
                 dense_units=250, p_drop=0.5):
        super().__init__()
        assert in_dim % timesteps == 0, f'{in_dim} not divisible by {timesteps}'
        self.timesteps = timesteps
        self.featurelen = in_dim // timesteps
        self.gru1 = nn.GRU(self.featurelen, units * 2, batch_first=True,
                            bidirectional=True)          # out: units*4
        self.drop1 = nn.Dropout(p_drop)
        self.t_att = TemporalAttention(timesteps)
        self.drop2 = nn.Dropout(p_drop)
        self.gru2 = nn.GRU(units * 4, units, batch_first=True,
                            bidirectional=True)          # out: units*2
        self.drop3 = nn.Dropout(p_drop)
        self.s_att = SpatialAttention(units * 2)
        self.drop4 = nn.Dropout(p_drop)
        self.head = ClassifierHead(units * 2, timesteps, dense_units, n_states, p_drop)

    def forward(self, x):
        x = _reshape_input(x, self.timesteps, self.featurelen)
        x, _ = self.gru1(x)                 # (B, T, units*4)
        x = self.drop1(x)
        x = self.t_att(x)
        x = self.drop2(x)
        x, _ = self.gru2(x)                 # (B, T, units*2)
        x = self.drop3(x)
        x = self.s_att(x)
        x = self.drop4(x)
        return self.head(x)


# ---------------------------------------------------------------------------
# 7. Full model with attention order swapped:
#    GRU -> spatial attention -> GRU -> temporal attention
# ---------------------------------------------------------------------------

class AE_BiLSTM_Swapped(nn.Module):
    def __init__(self, in_dim=2775, n_states=86, timesteps=5, units=250,
                 dense_units=250, p_drop=0.5):
        super().__init__()
        assert in_dim % timesteps == 0, f'{in_dim} not divisible by {timesteps}'
        self.timesteps = timesteps
        self.featurelen = in_dim // timesteps
        self.gru1 = nn.GRU(self.featurelen, units * 2, batch_first=True,
                            bidirectional=True)          # out: units*4
        self.drop1 = nn.Dropout(p_drop)
        self.s_att = SpatialAttention(units * 4)
        self.drop2 = nn.Dropout(p_drop)
        self.gru2 = nn.GRU(units * 4, units, batch_first=True,
                            bidirectional=True)          # out: units*2
        self.drop3 = nn.Dropout(p_drop)
        self.t_att = TemporalAttention(timesteps)
        self.drop4 = nn.Dropout(p_drop)
        self.head = ClassifierHead(units * 2, timesteps, dense_units, n_states, p_drop)

    def forward(self, x):
        x = _reshape_input(x, self.timesteps, self.featurelen)
        x, _ = self.gru1(x)                 # (B, T, units*4)
        x = self.drop1(x)
        x = self.s_att(x)
        x = self.drop2(x)
        x, _ = self.gru2(x)                 # (B, T, units*2)
        x = self.drop3(x)
        x = self.t_att(x)
        x = self.drop4(x)
        return self.head(x)


# ---------------------------------------------------------------------------
# Registry for looping over all variants in an experiment script
# ---------------------------------------------------------------------------

ABLATION_MODELS = {
    "single_gru":            SingleGRU,
    "single_bigru":           SingleBiGRU,
    "bigru_temporal_only":    BiGRU_TemporalAttOnly,
    "bigru_spatial_only":     BiGRU_SpatialAttOnly,
    "stacked_bigru_no_att":   StackedBiGRU_NoAtt,
    "full_model":             AE_BiLSTM,
    "full_model_swapped":     AE_BiLSTM_Swapped,
}


if __name__ == "__main__":
    # quick smoke test: run a dummy batch through every variant and print param counts
    x = torch.randn(4, 2775)
    for name, cls in ABLATION_MODELS.items():
        model = cls()
        y = model(x)
        n_params = sum(p.numel() for p in model.parameters())
        print(f"{name:22s} out={tuple(y.shape)}  params={n_params/1e6:.2f}M")

In [ ]:
# in_dim = Feature_train.shape[1] 
# n_states = 86
# timesteps = 5

# os.makedirs("/kaggle/working/Online-Recog", exist_ok=True)
# ablation_results = {}

# for name, ModelClass in ABLATION_MODELS.items():
#     model = ModelClass(in_dim=in_dim, n_states=n_states, timesteps=timesteps).to(device)
#     print("---------------------Training " + name + " model -------------------------------")
#     model, history =train_model(model, Feature_train, Targets_train, Feature_val, Targets_val,
#                   epochs=15, batch_size=64, lr=1e-3, weight_decay=1e-3, device=device, seed=42, factor=0.5)
#     results, predictions, targets = evaluate_online(model, parsers, scaler, Prior, Transition_matrix, bilstm=False)
#     path_emission = "/kaggle/working/Online-Recog/emission_" + name + ".pt"
#     save_emission(model, path_emission)
#     path_history = "/kaggle/working/Online-Recog/history_" + name + ".npy"
#     np.save(path_history, history)

#     # store everything for later plotting/analysis
#     ablation_results[name] = {
#         "history": history,          # dict with e.g. 'loss', 'val_loss', 'train_acc', 'val_acc', ...
#         "results": results,          # online recognition metrics (recall/precision/F1/etc.)
#         "predictions": predictions,
#         "targets": targets,
#         "n_params": sum(p.numel() for p in model.parameters()),
#         "emission_path": path_emission,
#         "history_path": path_history,
#     }
 
#     print(f"{name}: done.")

# np.save("/kaggle/working/Online-Recog/ablation_results_all.npy", ablation_results, allow_pickle=True)

In [ ]:
# def _get_loss_keys(history):
#     """Best-effort lookup for train/val loss keys, since naming can vary
#     slightly depending on how train_model logs its history dict."""
#     train_key_candidates = ["loss", "train_loss"]
#     val_key_candidates = ["val_loss", "valid_loss"]
 
#     train_key = next((k for k in train_key_candidates if k in history), None)
#     val_key = next((k for k in val_key_candidates if k in history), None)
 
#     if train_key is None or val_key is None:
#         raise KeyError(
#             f"Could not find loss keys in history. Available keys: {list(history.keys())}. "
#             f"Update train_key_candidates/val_key_candidates in _get_loss_keys() to match."
#         )
#     return train_key, val_key
 
 
# def plot_ablation_losses(ablation_results, figsize=(14, 5), save_path=None):
#     """Plots train loss and val loss side by side, one line per model."""
#     fig, (ax_train, ax_val) = plt.subplots(1, 2, figsize=figsize)
 
#     for name, entry in ablation_results.items():
#         history = entry["history"]
#         train_key, val_key = _get_loss_keys(history)
#         train_loss = history[train_key]
#         val_loss = history[val_key]
#         epochs = range(1, len(train_loss) + 1)
 
#         ax_train.plot(epochs, train_loss, label=name, marker='o', markersize=3)
#         ax_val.plot(epochs, val_loss, label=name, marker='o', markersize=3)
 
#     ax_train.set_title("Train Loss")
#     ax_train.set_xlabel("Epoch")
#     ax_train.set_ylabel("Loss")
#     ax_train.legend(fontsize=8)
#     ax_train.grid(alpha=0.3)
 
#     ax_val.set_title("Validation Loss")
#     ax_val.set_xlabel("Epoch")
#     ax_val.set_ylabel("Loss")
#     ax_val.legend(fontsize=8)
#     ax_val.grid(alpha=0.3)
 
#     fig.tight_layout()
#     if save_path:
#         fig.savefig(save_path, dpi=150, bbox_inches="tight")
#     plt.show()

 
 
# # usage:
# plot_ablation_losses(ablation_results, save_path="/kaggle/working/Online-Recog/ablation_loss_curves.png")

# Plots

In [ ]:
# def plot_histories(histories, save_path="/kaggle/working/Online-Recog/training-OR.jpeg"):
#     """
#     histories: dict of {model_name: history_dict}, where each history_dict
#         has keys 'epoch', 'train_loss', 'val_loss', 'train_acc', 'val_acc'
#         (as produced by train_model/train_emission).

#     Produces a 1x2 grid:
#         left  panel: train + val loss curves for every model
#         right panel: train + val accuracy curves for every model

#     Solid lines = train, dashed lines = val. Each model gets its own color,
#     consistent across both panels and both metrics.
#     """
#     fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(13, 5))

#     colors = plt.get_cmap('tab10').colors
#     for idx, (name, hist) in enumerate(histories.items()):
#         color = colors[idx % len(colors)]
#         epochs = hist['epoch']

#         ax_loss.plot(epochs, hist['train_loss'], color=color, linestyle='-',
#                      label=f'{name} train')
#         ax_loss.plot(epochs, hist['val_loss'], color=color, linestyle='--',
#                      label=f'{name} val')

#         ax_acc.plot(epochs, hist['train_acc'], color=color, linestyle='-',
#                     label=f'{name} train')
#         ax_acc.plot(epochs, hist['val_acc'], color=color, linestyle='--',
#                     label=f'{name} val')

#     ax_loss.set_xlabel('epoch'); ax_loss.set_ylabel('loss')
#     ax_loss.set_title('Training / validation loss')
#     ax_loss.legend(fontsize=8); ax_loss.grid(alpha=0.3)

#     ax_acc.set_xlabel('epoch'); ax_acc.set_ylabel('accuracy')
#     ax_acc.set_title('Training / validation accuracy')
#     ax_acc.legend(fontsize=8); ax_acc.grid(alpha=0.3)

#     plt.tight_layout()
#     if save_path:
#         plt.savefig(save_path, dpi=200)
#     plt.show()
#     return fig, (ax_loss, ax_acc)


# # usage
# hist_mlp = np.load("/kaggle/input/datasets/dandrade05/results-or/history_MLP.npy", allow_pickle=True).item()
# hist_bilstm = np.load("/kaggle/input/datasets/dandrade05/results-or/history_bilstm.npy", allow_pickle=True).item()
# plot_histories({"MLP": hist_mlp, "AED-BiLSTM": hist_bilstm})

# Exploring predictions

In [14]:
def load_events(path, is_target=False):
    """Return list of (label, begin_frame, end_frame) tuples. (Legacy loader
    for the single-flat-array .npy format from before.)"""
    arr = np.load(path, allow_pickle=True)
    if is_target:
        return [(int(c), int(b), int(e)) for c, b, e in arr]           # (N,3)
    labels, begins, ends = arr[0], arr[1], arr[2]                       # (3,N)
    return [(int(c), int(b), int(e)) for c, b, e in zip(labels, begins, ends)]


def load_targets_zip(targets_path):
    """
    Load a targets.npy that is already an array of 5 subjects, each a list
    of [label, begin, end] triples (this is the format used inside
    preds_MLP2.zip's targets.npy). Returns exactly that: a list of 5
    subject-lists of [label, begin, end].
    """
    arr = np.load(targets_path, allow_pickle=True)
    return [[[int(label), int(b), int(e)] for label, b, e in subject]
            for subject in arr]


def load_predictions_zip(predictions_path):
    """
    Load a predictions.npy shaped (n_subjects, 4), where each row is
    [sample_id, label_array, begin_array, end_array] (the format used
    inside preds_MLP2.zip's predictions.npy). Converts each subject's three
    parallel arrays into a list of [label, begin, end] triples, giving the
    same list-of-subjects-of-triples shape evaluate_per_class expects.
    """
    arr = np.load(predictions_path, allow_pickle=True)
    subjects = []
    for row in arr:
        _sample_id, labels, begins, ends = row
        subjects.append([[int(label), int(b), int(e)]
                          for label, b, e in zip(labels, begins, ends)])
    return subjects


def evaluate_per_class(targets, preds, threshold_alpha=0.3, classes=None):
    """
    Direct per-class extension of the scoring block from evaluate_online:

        for gt_label, gt_b, gt_e in gesturesList:
            for i in range(len(begin_frame)):
                if pred_label[i] == gt_label:
                    inter = min(end_frame[i], gt_e) - max(begin_frame[i], gt_b)
                    union = max(end_frame[i], gt_e) - min(begin_frame[i], gt_b)
                    if union > 0 and inter / union > threshold_alpha:
                        correct += 1; break

    Same "same label + IoU > threshold_alpha" matching rule, same accounting
    (recall over ground-truth events, precision over predicted events),
    just tallied per class instead of pooled.

    `targets` and `preds` are both lists with one element per subject
    (same length, same subject order), where each element is itself a list
    of [label, begin_frame, end_frame] events for that subject, e.g.:

        targets = [
            [[8, 66, 89], [9, 120, 140], ...],   # subject 1
            [[8, 40, 70], ...],                   # subject 2
            ...
        ]
        preds = [
            [[8, 70, 92], ...],                   # subject 1 predictions
            ...
        ]

    Matching is done per subject (a prediction can only match ground-truth
    events from the *same* subject), then counts are pooled across subjects
    before computing per-class recall/precision/F1 - mirroring how
    evaluate_online accumulates `correct`/`recall_len`/`pred_len_tot` while
    looping over `Sample_list`.
    """
    assert len(targets) == len(preds), "targets and preds must have the same number of subjects"

    if classes is None:
        classes = sorted(set(g[0] for gesturesList in targets for g in gesturesList))

    correct = {c: 0 for c in classes}
    recall_len = {c: 0 for c in classes}
    pred_len_tot = {c: 0 for c in classes}

    for gesturesList, subj_preds in zip(targets, preds):
        pred_label = [p[0] for p in subj_preds]
        begin_frame = [p[1] for p in subj_preds]
        end_frame = [p[2] for p in subj_preds]

        for gt_label, gt_b, gt_e in gesturesList:
            if gt_label not in recall_len:
                continue
            recall_len[gt_label] += 1
            for i in range(len(begin_frame)):
                if pred_label[i] == gt_label:
                    inter = min(end_frame[i], gt_e) - max(begin_frame[i], gt_b)
                    union = max(end_frame[i], gt_e) - min(begin_frame[i], gt_b)
                    if union > 0 and inter / union > threshold_alpha:
                        correct[gt_label] += 1
                        break

        for label in pred_label:
            if label in pred_len_tot:
                pred_len_tot[label] += 1

    rows = []
    for c in classes:
        r = correct[c] / recall_len[c] if recall_len[c] > 0 else 0.0
        p = correct[c] / pred_len_tot[c] if pred_len_tot[c] > 0 else 0.0
        f1 = 2 * r * p / (r + p + 1e-7) if (r + p) > 0 else 0.0
        rows.append({
            "class": c,
            "correct": correct[c],
            "n_gt": recall_len[c],
            "n_pred": pred_len_tot[c],
            "recall": round(r, 4),
            "precision": round(p, 4),
            "f1_score": round(f1, 4),
        })

    df = pd.DataFrame(rows)

    total_correct = sum(correct.values())
    total_gt = sum(recall_len.values())
    total_pred = sum(pred_len_tot.values())
    overall_recall = total_correct / max(total_gt, 1)
    overall_precision = total_correct / max(total_pred, 1)
    overall_f1 = 2 * overall_recall * overall_precision / (overall_recall + overall_precision + 1e-7)

    return df, dict(recall=overall_recall, precision=overall_precision, f1=overall_f1)

In [15]:
if __name__ == "__main__":
    THRESHOLD_ALPHA = 0.3  # same default as evaluate_online

    # --- new format: preds_<model>.zip, each containing targets.npy (5
    # subjects x list of [label,begin,end]) and predictions.npy (5 subjects
    # x [sample_id, label_arr, begin_arr, end_arr]). ---
    import zipfile

    model_zips = {
        "MLP": "/kaggle/input/datasets/dandrade05/predictions/preds_MLP/predictions.npy",
        "MLP2": "/kaggle/input/datasets/dandrade05/predictions/preds_MLP2/predictions.npy",
        "BiLSTM": "/kaggle/input/datasets/dandrade05/predictions/preds_bilstm/predictions.npy",
        "BiLSTM (weighted)": "/kaggle/input/datasets/dandrade05/predictions/preds_bilstm_weighted/predictions.npy",

    }

    targets = None  # shared across models - assumed identical in every zip
    all_preds = {}
    for name, zip_path in model_zips.items():
            # extract_dir = f"/home/claude/{name.replace(' ', '_')}_extract"
            # with zipfile.ZipFile(zip_path) as zf:
            #     zf.extractall(extract_dir)

        subj_targets = load_targets_zip(f"/kaggle/input/datasets/dandrade05/predictions/preds_MLP/targets.npy")
        if targets is None:
            targets = subj_targets
        all_preds[name] = load_predictions_zip(zip_path)

    # union of ground-truth classes and every class any model predicted, so
    # a class predicted but never in targets (e.g. a spurious label) still
    # shows up as pure FP instead of silently disappearing
    target_classes = set(g[0] for subject in targets for g in subject)
    pred_classes = set(p[0] for preds in all_preds.values() for subject in preds for p in subject)
    classes = sorted(target_classes | pred_classes)

    for name, preds in all_preds.items():
        df, overall = evaluate_per_class(targets, preds, threshold_alpha=THRESHOLD_ALPHA, classes=classes)

        print(f"\n=== {name} (threshold_alpha={THRESHOLD_ALPHA}) ===")
        print(df.to_string(index=False))
        print(f"Overall -> recall={overall['recall']:.4f}  "
              f"precision={overall['precision']:.4f}  f1={overall['f1']:.4f}")


=== MLP (threshold_alpha=0.3) ===
 class  correct  n_gt  n_pred  recall  precision  f1_score
     0        0    13       0  0.0000     0.0000    0.0000
     1        0     3       0  0.0000     0.0000    0.0000
     2        0    15       0  0.0000     0.0000    0.0000
     3        0     2       0  0.0000     0.0000    0.0000
     4        0    49       0  0.0000     0.0000    0.0000
     5        0     1       0  0.0000     0.0000    0.0000
     6        0     3       0  0.0000     0.0000    0.0000
     7        0   191       0  0.0000     0.0000    0.0000
     8       18   243      53  0.0741     0.3396    0.1216
     9        0    34       0  0.0000     0.0000    0.0000
    10        0     8       0  0.0000     0.0000    0.0000
    11        1    30       9  0.0333     0.1111    0.0513
    12        0     4       0  0.0000     0.0000    0.0000
    13        0     5       0  0.0000     0.0000    0.0000
    14        0     0       0  0.0000     0.0000    0.0000
    15        0     0